# Multimodal 3D Generation - Google Colab Training (CPU)

Notebook này chạy **hoàn toàn trên CPU** (RAM 12.7 GB, Disk 225.8 GB):
1. Mount Google Drive + kiểm tra project code
2. Tải dữ liệu **ABO + ShapeNet** (~15 GB thêm) → lưu trên Drive
3. **Phân tích dataset** (thống kê captions, categories)
4. **Áp dụng MatAlpha mask** lọc background (Phase 1)
5. Cấu hình training **CPU-only** (MLP decoder, batch=8, ~10 phút/epoch)
6. Train model với **auto-save mỗi 30 batches** → resume tự động
7. Evaluate metrics (Chamfer Distance, F-Score)
8. Demo visualization

### Bảo hiểm chống ngắt:
- ✅ Checkpoint lưu **mỗi 30 batches** (~36 giây progress tối đa)
- ✅ **Resume tự động** khi reconnect
- ✅ Keep-alive JavaScript chống timeout
- ✅ Atomic save (write .tmp → rename) chống file hỏng
- ✅ Dữ liệu trên Drive → không mất khi session reset

## 1. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    print('✓ Google Drive mounted')
except:
    print('✓ Google Drive already mounted')

# Xác nhận CPU-only
import torch, psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'\n💻 Chạy trên CPU (RAM: {ram_gb:.1f} GB)')
print(f'   Config: MLP decoder, batch=8, max_objects=5000')
print(f'   1 epoch ≈ 8-15 phút')

In [ ]:
import os

# === CẤU HÌNH ĐƯỜNG DẪN ===
# Mọi thứ chạy trực tiếp trên Drive
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'

# Tạo thư mục nếu chưa có
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

print(f'✓ Work dir: {WORK_DIR}')

## 2. Kiểm tra Project Code

Project code nằm trực tiếp trên Drive.  
Đảm bảo đã upload các thư mục sau lên Drive: `src/`, `configs/`, `images/`

In [ ]:
# Kiểm tra xem có đủ files không
import os

required_files = {
    'src/__init__.py': 'Package init',
    'src/train.py': 'Training script',
    'src/evaluate.py': 'Evaluation script',
    'src/demo.py': 'Gradio demo',
    'src/data/__init__.py': 'Data package init',
    'src/data/dataset.py': 'Dataset & DataLoader',
    'src/data/prepare_data.py': 'Data preparation',
    'src/data/transforms.py': 'Data transforms',
    'src/models/__init__.py': 'Models package init',
    'src/models/multimodal3d.py': 'Main model',
    'src/models/encoders.py': 'CLIP encoders',
    'src/models/fusion.py': 'Fusion modules',
    'src/models/decoder.py': 'Point cloud decoder',
    'src/losses/__init__.py': 'Losses package init',
    'src/losses/losses.py': 'Loss functions',
    'configs/colab.yaml': 'Default config (PC)',
}

all_ok = True
for rel_path, desc in required_files.items():
    path = os.path.join(WORK_DIR, rel_path)
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ THIẾU'
    print(f'  {status}: {rel_path} ({desc})')
    if not exists:
        all_ok = False

if all_ok:
    print(f'\n✓ Project structure OK! ({len(required_files)} files)')
else:
    print('\n✗ Thiếu files! Upload toàn bộ thư mục src/ và configs/ lên Drive.')
    print(f'  Drive path: {WORK_DIR}/')

## 3. Cài Dependencies

In [ ]:
# Cài các thư viện cần thiết
!pip install -q open-clip-torch trimesh gradio plotly tensorboard scipy tqdm pyyaml open3d psutil

# Verify imports
import torch, open_clip, trimesh, plotly, scipy, psutil
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
try:
    import open3d
    print(f'Open3D: {open3d.__version__} (mesh chất lượng cao)')
except:
    print('Open3D: không cài được (dùng trimesh fallback)')

## 4. Tải dữ liệu Cap3D

Dữ liệu sẽ được tải và giải nén trực tiếp lên Drive (~40GB).  
Nếu đã có data, cell này sẽ skip.

In [ ]:
%%time
import os, zipfile, shutil, io, json

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
os.makedirs(f'{DATA_DIR}/pointclouds', exist_ok=True)
os.makedirs(f'{DATA_DIR}/rendered_images', exist_ok=True)

# --- [1/3] Captions ABO ---
csv_path = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
if os.path.exists(csv_path):
    print('[1/3] ✓ Captions ABO đã có')
else:
    print('[1/3] Tải captions ABO...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D Cap3D_automated_ABO.csv \
        --repo-type dataset --local-dir /tmp/cap3d_download
    shutil.copy2('/tmp/cap3d_download/Cap3D_automated_ABO.csv', csv_path)
    print('✓ Captions saved')

# --- [2/3] Point clouds ABO ---
pc_dir = f'{DATA_DIR}/pointclouds'
n_pc = len([f for f in os.listdir(pc_dir) if f.endswith('.ply')])
if n_pc > 100:
    print(f'\n[2/3] ✓ Point clouds ABO đã có ({n_pc} files)')
else:
    print('\n[2/3] Tải point clouds ABO (~1.5GB)...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D PointCloud_zips_ABO/compressed_pcs_00.zip \
        --repo-type dataset --local-dir /tmp/cap3d_download
    pc_zip = '/tmp/cap3d_download/PointCloud_zips_ABO/compressed_pcs_00.zip'
    print('Giải nén lên Drive...')
    with zipfile.ZipFile(pc_zip, 'r') as z:
        members = [m for m in z.namelist() if m.endswith('.ply')]
        for i, m in enumerate(members, 1):
            fname = os.path.basename(m)
            with z.open(m) as src, open(f'{pc_dir}/{fname}', 'wb') as dst:
                dst.write(src.read())
            if i % 2000 == 0 or i == len(members):
                print(f'  [{i}/{len(members)}]')
    os.remove(pc_zip)
    print(f'✓ Point clouds ABO: {len(os.listdir(pc_dir))} files')

# --- [3/3] Rendered images ABO (SMART RESUME) ---
img_dir = f'{DATA_DIR}/rendered_images'
existing_folders = set([d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')])

import pandas as pd
df = pd.read_csv(csv_path, header=None)
expected_ids = set(df[0].astype(str).tolist())
print(f'ABO objects: {len(expected_ids)}')

missing_ids = expected_ids - existing_folders
n_missing = len(missing_ids)
n_total = len(expected_ids)

if n_missing == 0:
    print(f'\n[3/3] ✓ Rendered images đã có ({len(existing_folders)}/{n_total} folders)')
elif n_missing < 10:
    print(f'\n[3/3] Rendered images thiếu {n_missing}/{n_total} folders (ít, bỏ qua)')
else:
    print(f'\n[3/3] Rendered images thiếu ({n_missing}/{n_total} folders). Tiếp tục tải...')
    !pip install -q huggingface_hub
    
    main_zip_path = '/tmp/cap3d_download/RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip'
    if not os.path.exists(main_zip_path):
        print('Đang tải main zip (~37GB)...')
        !hf download tiange/Cap3D RenderedImage_perobj_zips_ABO/compressed_imgs_perobj_00.zip \
            --repo-type dataset --local-dir /tmp/cap3d_download
        print('✓ Main zip downloaded')
    
    print(f'Giải nén {n_missing} folders còn thiếu...')
    with zipfile.ZipFile(main_zip_path, 'r') as outer_z:
        inner_zips = [m for m in outer_z.namelist() if m.endswith('.zip')]
        zip_to_id = {z: os.path.basename(z).replace('.zip', '') for z in inner_zips}
        needed_zips = [z for z in inner_zips if zip_to_id[z] in missing_ids]
        for i, inner_name in enumerate(needed_zips, 1):
            obj_id = zip_to_id[inner_name]
            obj_dir = f'{img_dir}/{obj_id}'
            os.makedirs(obj_dir, exist_ok=True)
            inner_data = outer_z.read(inner_name)
            with zipfile.ZipFile(io.BytesIO(inner_data)) as inner_z:
                for img_file in inner_z.namelist():
                    if img_file.endswith(('.png', '.jpg')):
                        fname = os.path.basename(img_file)
                        with inner_z.open(img_file) as src, open(f'{obj_dir}/{fname}', 'wb') as dst:
                            dst.write(src.read())
            if i % 100 == 0 or i == len(needed_zips):
                print(f'  [{i}/{len(needed_zips)}]')
    
    if os.path.exists(main_zip_path):
        os.remove(main_zip_path)

# --- Tạo splits (ABO + ShapeNet nếu có) ---
print('\nTạo splits...')
os.chdir(WORK_DIR)
splits_dir = f'{DATA_DIR}/splits'
if os.path.exists(splits_dir):
    shutil.rmtree(splits_dir)

# Kiểm tra ShapeNet point clouds
sn_pc_dir = f'{DATA_DIR}/shapenet/pointclouds'
has_shapenet = os.path.exists(sn_pc_dir) and len([f for f in os.listdir(sn_pc_dir) if f.endswith('.ply')]) > 100

shapenet_flag = '--include-shapenet' if has_shapenet else ''
if has_shapenet:
    n_sn = len([f for f in os.listdir(sn_pc_dir) if f.endswith('.ply')])
    print(f'  ✓ ShapeNet point clouds tìm thấy ({n_sn} files) → gộp vào splits')
else:
    print('  ShapeNet point clouds chưa có → chỉ dùng ABO')

!python src/data/prepare_data.py --data-dir "{DATA_DIR}" --skip-extract {shapenet_flag}

for split in ['train', 'val', 'test']:
    sp = f'{DATA_DIR}/splits/{split}.txt'
    if os.path.exists(sp):
        with open(sp) as f:
            n = len([l for l in f if l.strip()])
        print(f'  {split}: {n} objects')

!df -h /content | tail -1
print('\n✓ Data sẵn sàng!')

## 4.2 Tải thêm ShapeNet (Phase 2 - Plan.md)

ShapeNet có **52,472 objects** → gấp 7 lần ABO.  
Chỉ tải **point clouds + captions** (~15 GB). Không tải rendered images (~30 GB) để tiết kiệm thời gian.

**Disk 225.8 GB cho phép tải thêm ShapeNet.**

In [ ]:
%%time
import os, zipfile, shutil

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
SHAPENET_DIR = f'{DATA_DIR}/shapenet'
os.makedirs(f'{SHAPENET_DIR}/pointclouds', exist_ok=True)

# === [1/2] ShapeNet Captions ===
shapenet_csv = f'{DATA_DIR}/Cap3D_automated_ShapeNet.csv'
if os.path.exists(shapenet_csv):
    print('[1/2] ✓ ShapeNet captions đã có')
else:
    print('[1/2] Tải ShapeNet captions (~19.5 MB)...')
    !pip install -q huggingface_hub
    !hf download tiange/Cap3D Cap3D_automated_ShapeNet.csv \
        --repo-type dataset --local-dir /tmp/cap3d_shapenet
    shutil.copy2('/tmp/cap3d_shapenet/Cap3D_automated_ShapeNet.csv', shapenet_csv)
    print('✓ ShapeNet captions saved')

# === [2/2] ShapeNet Point Clouds (~15 GB, 1 file) ===
shapenet_pc_dir = f'{SHAPENET_DIR}/pointclouds'
n_pc = len([f for f in os.listdir(shapenet_pc_dir) if f.endswith('.ply')])

if n_pc > 10000:
    print(f'\n[2/2] ✓ ShapeNet point clouds đã có ({n_pc} files)')
else:
    print(f'\n[2/2] ShapeNet point clouds ({n_pc} files hiện có). Tải thêm...')
    !pip install -q huggingface_hub
    
    zip_name = 'compressed_pcs_00.zip'
    zip_path = f'/tmp/cap3d_shapenet/PointCloud_zips_ShapeNet/{zip_name}'
    marker = f'{shapenet_pc_dir}/.extracted_00'
    
    if os.path.exists(marker):
        print(f'  ✓ {zip_name} đã extract')
    else:
        if not os.path.exists(zip_path):
            print(f'  Tải {zip_name} (~15 GB)...')
            !hf download tiange/Cap3D PointCloud_zips_ShapeNet/{zip_name} \
                --repo-type dataset --local-dir /tmp/cap3d_shapenet
        
        print(f'  Giải nén {zip_name}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            members = [m for m in z.namelist() if m.endswith('.ply')]
            for i, m in enumerate(members, 1):
                fname = os.path.basename(m)
                with z.open(m) as src, open(f'{shapenet_pc_dir}/{fname}', 'wb') as dst:
                    dst.write(src.read())
                if i % 5000 == 0 or i == len(members):
                    print(f'    [{i}/{len(members)}]')
        
        open(marker, 'w').close()
        os.remove(zip_path)
        print(f'  ✓ {zip_name} done, deleted zip')

    n_final = len([f for f in os.listdir(shapenet_pc_dir) if f.endswith('.ply')])
    print(f'\n✓ ShapeNet point clouds: {n_final} files')

# === Thống kê tổng ===
import pandas as pd
abo_csv = f'{DATA_DIR}/Cap3D_automated_ABO.csv'
n_abo = len(pd.read_csv(abo_csv, header=None))
n_shapenet = len(pd.read_csv(shapenet_csv, header=None))
print(f'\n=== TỔNG DỮ LIỆU ===')
print(f'  ABO:      {n_abo:,} objects')
print(f'  ShapeNet: {n_shapenet:,} objects')
print(f'  Tổng:     {n_abo + n_shapenet:,} objects')
!df -h /content/drive | tail -1

## 4.3 Convert Point Cloud .ply → .pt (tăng tốc loading 5-10x)

Chạy **1 lần duy nhất**. Sau khi convert xong, training sẽ tự động dùng .pt (nhanh hơn trimesh rất nhiều trên CPU).

In [ ]:
%%time
import torch, trimesh, os, glob, numpy as np

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
ply_dir = f'{DATA_DIR}/pointclouds'
pt_dir = f'{DATA_DIR}/pointclouds_pt'
os.makedirs(pt_dir, exist_ok=True)

# Kiểm tra xem đã convert chưa
existing_pt = len([f for f in os.listdir(pt_dir) if f.endswith('.pt')])
existing_ply = len([f for f in os.listdir(ply_dir) if f.endswith('.ply')])

if existing_pt >= existing_ply and existing_pt > 100:
    print(f'✓ Đã convert: {existing_pt} files .pt (skip)')
else:
    print(f'Converting {existing_ply} files .ply → .pt ...')
    converted = 0
    errors = 0
    
    for ply_file in sorted(glob.glob(f'{ply_dir}/*.ply')):
        obj_id = os.path.basename(ply_file).replace('.ply', '')
        pt_path = f'{pt_dir}/{obj_id}.pt'
        
        if os.path.exists(pt_path):
            converted += 1
            continue
        
        try:
            mesh = trimesh.load(ply_file)
            xyz = torch.tensor(mesh.vertices, dtype=torch.float32)
            
            if hasattr(mesh, 'visual') and hasattr(mesh.visual, 'vertex_colors'):
                vc = mesh.visual.vertex_colors
                if vc is not None and len(vc) == len(xyz):
                    rgb = torch.tensor(vc[:, :3], dtype=torch.float32) / 127.5 - 1.0
                else:
                    rgb = torch.zeros_like(xyz)
            else:
                rgb = torch.zeros_like(xyz)
            
            pc = torch.cat([xyz, rgb], dim=1)  # [N, 6]
            torch.save(pc, pt_path)
            converted += 1
        except Exception as e:
            errors += 1
        
        if converted % 1000 == 0:
            print(f'  [{converted}/{existing_ply}] converted, {errors} errors')
    
    print(f'\n✓ Convert hoàn tất: {converted} files .pt, {errors} errors')
    print(f'  Training sẽ tự động dùng .pt (5-10x nhanh hơn .ply)')

# Thống kê dung lượng
pt_size = sum(os.path.getsize(f'{pt_dir}/{f}') for f in os.listdir(pt_dir) if f.endswith('.pt'))
print(f'\n  Dung lượng .pt: {pt_size / (1024**3):.2f} GB')

## 4.7 Phân tích Dataset + Áp dụng MatAlpha Mask (Phase 1)

- Thống kê captions, categories từ ABO + ShapeNet
- Kiểm tra multi-view data (depth, mask)
- **Áp dụng MatAlpha mask** để lọc background trước khi train

In [ ]:
import os, json
import pandas as pd
from collections import Counter
import numpy as np
from PIL import Image

DATA_DIR = f'{WORK_DIR}/Cap3D_data'

# === 1. Thống kê captions (ABO + ShapeNet) ===
print('=== THỐNG KÊ DATASET ===')

# ABO captions
captions_path = f'{DATA_DIR}/captions.json'
if os.path.exists(captions_path):
    with open(captions_path) as f:
        abo_captions = json.load(f)
    print(f'ABO: {len(abo_captions)} objects')

# ShapeNet captions
shapenet_csv = f'{DATA_DIR}/Cap3D_automated_ShapeNet.csv'
if os.path.exists(shapenet_csv):
    df_sn = pd.read_csv(shapenet_csv, header=None)
    print(f'ShapeNet: {len(df_sn)} objects')
else:
    df_sn = pd.DataFrame()
    print('ShapeNet: chưa tải')

# Trích xuất từ khóa phổ biến
words = []
if os.path.exists(captions_path):
    for obj_id, caps in abo_captions.items():
        cap_text = caps.get('blip2', caps.get('caption', caps.get('general', '')))
        if isinstance(cap_text, list):
            cap_text = cap_text[0]
        words.extend(str(cap_text).lower().split())
if len(df_sn) > 0:
    for _, row in df_sn.iterrows():
        words.extend(str(row[1]).lower().split())

counter = Counter(words)
stopwords = {'a', 'an', 'the', 'and', 'or', 'with', 'of', 'in', 'on', 'is', 'it', 'to', 'for', '3d'}
keywords = [(w, c) for w, c in counter.most_common(100) if w not in stopwords and len(w) > 2][:20]

print(f'\n--- Top 20 từ khóa (ABO + ShapeNet) ---')
for i, (word, count) in enumerate(keywords, 1):
    bar = '█' * min(count // 100, 30)
    print(f'  {i:2d}. {word:15s} ({count:5d}) {bar}')

# === 2. Kiểm tra rendered images (multi-view, depth, mask) ===
img_dir = f'{DATA_DIR}/rendered_images'
if os.path.exists(img_dir):
    sample_dirs = [d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')][:3]
    print(f'\n--- Rendered images (mẫu) ---')
    has_mask = False
    for d in sample_dirs:
        files = os.listdir(f'{img_dir}/{d}')
        rgb_files = [f for f in files if f.endswith('.png') and '_depth' not in f and '_MatAlpha' not in f]
        depth_files = [f for f in files if '_depth' in f]
        mask_files = [f for f in files if '_MatAlpha' in f]
        has_mask = has_mask or len(mask_files) > 0
        print(f'  {d}: {len(rgb_files)} RGB, {len(depth_files)} depth, {len(mask_files)} mask')
    
    if has_mask:
        print('\n✓ MatAlpha masks có sẵn → sẽ áp dụng lọc background')
    else:
        print('\n⚠ Không tìm thấy MatAlpha masks')

# === 3. Áp dụng MatAlpha Mask (Phase 1 - Plan.md) ===
# Tạo thư mục chứa ảnh đã lọc background
MASKED_DIR = f'{DATA_DIR}/rendered_images_masked'
os.makedirs(MASKED_DIR, exist_ok=True)

# Kiểm tra xem đã preprocess chưa
existing_masked = len([d for d in os.listdir(MASKED_DIR) if os.path.isdir(f'{MASKED_DIR}/{d}')]) if os.path.exists(MASKED_DIR) else 0

if existing_masked > 100:
    print(f'\n✓ MatAlpha mask đã áp dụng ({existing_masked} folders)')
else:
    print(f'\n--- Áp dụng MatAlpha mask lọc background ---')
    all_obj_dirs = [d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')]
    processed = 0
    
    for obj_id in all_obj_dirs:
        obj_img_dir = f'{img_dir}/{obj_id}'
        obj_masked_dir = f'{MASKED_DIR}/{obj_id}'
        
        if os.path.exists(obj_masked_dir):
            processed += 1
            continue
        
        os.makedirs(obj_masked_dir, exist_ok=True)
        files = os.listdir(obj_img_dir)
        rgb_files = sorted([f for f in files if f.endswith('.png') and '_depth' not in f and '_MatAlpha' not in f])
        
        for rgb_name in rgb_files:
            rgb_path = f'{obj_img_dir}/{rgb_name}'
            mask_name = rgb_name.replace('.png', '_MatAlpha.png')
            mask_path = f'{obj_img_dir}/{mask_name}'
            
            rgb = Image.open(rgb_path).convert('RGB')
            
            if os.path.exists(mask_path):
                mask = Image.open(mask_path).convert('L')
                mask_arr = np.array(mask) > 128
                img_arr = np.array(rgb)
                img_arr[~mask_arr] = [255, 255, 255]  # Background = trắng
                rgb = Image.fromarray(img_arr)
            
            rgb.save(f'{obj_masked_dir}/{rgb_name}')
        
        processed += 1
        if processed % 500 == 0:
            print(f'  [{processed}/{len(all_obj_dirs)}] objects processed')
    
    print(f'✓ MatAlpha mask applied: {processed} objects → {MASKED_DIR}')
    print('  Background = trắng (tối ưu cho CLIP encoding)')

print(f'\n✓ Phân tích + preprocessing hoàn tất!')

## 4.8 Chuẩn hóa Caption sang Tiếng Việt + Thống kê nhãn (note.txt)

Dịch captions sang tiếng Việt bằng Google Translate API, sau đó thống kê lại số nhãn theo category.

In [ ]:
!pip install -q googletrans==4.0.0-rc1

import os, json
import pandas as pd
from collections import Counter

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
VI_CAPTIONS_PATH = f'{DATA_DIR}/captions_vi.json'

# === 1. Dịch captions sang tiếng Việt ===
if os.path.exists(VI_CAPTIONS_PATH):
    with open(VI_CAPTIONS_PATH) as f:
        vi_captions = json.load(f)
    print(f'✓ Captions tiếng Việt đã có ({len(vi_captions)} objects)')
else:
    print('Đang dịch captions sang tiếng Việt...')
    from googletrans import Translator
    translator = Translator()
    
    captions_path = f'{DATA_DIR}/captions.json'
    with open(captions_path) as f:
        all_captions = json.load(f)
    
    vi_captions = {}
    total = len(all_captions)
    
    for i, (obj_id, caps) in enumerate(all_captions.items(), 1):
        cap_text = caps.get('blip2', caps.get('caption', caps.get('general', '')))
        if isinstance(cap_text, list):
            cap_text = cap_text[0]
        
        try:
            result = translator.translate(str(cap_text), src='en', dest='vi')
            vi_captions[obj_id] = {
                'en': str(cap_text),
                'vi': result.text
            }
        except Exception as e:
            vi_captions[obj_id] = {
                'en': str(cap_text),
                'vi': str(cap_text)  # Giữ nguyên nếu lỗi
            }
        
        if i % 500 == 0 or i == total:
            print(f'  [{i}/{total}] objects translated')
            # Lưu intermediate để không mất progress
            with open(VI_CAPTIONS_PATH, 'w', encoding='utf-8') as f:
                json.dump(vi_captions, f, ensure_ascii=False, indent=2)
    
    print(f'✓ Đã dịch {len(vi_captions)} captions → {VI_CAPTIONS_PATH}')

# === 2. Hiển thị mẫu captions tiếng Việt ===
print('\n--- Mẫu captions tiếng Việt ---')
sample_items = list(vi_captions.items())[:10]
for obj_id, caps in sample_items:
    print(f'  {obj_id}:')
    print(f'    EN: {caps["en"][:80]}')
    print(f'    VI: {caps["vi"][:80]}')
    print()

# === 3. Thống kê nhãn (tiếng Việt) ===
print('--- Thống kê nhãn tiếng Việt ---')
vi_words = []
for obj_id, caps in vi_captions.items():
    vi_words.extend(caps['vi'].lower().split())

vi_counter = Counter(vi_words)
vi_stopwords = {'một', 'và', 'với', 'của', 'có', 'là', 'các', 'được', 'trong', 'trên', 'cho', 'từ', 'này'}
vi_keywords = [(w, c) for w, c in vi_counter.most_common(100) if w not in vi_stopwords and len(w) > 2][:20]

print(f'Tổng số từ: {len(vi_words):,}')
print(f'Số từ khác nhau: {len(vi_counter):,}')
print(f'\nTop 20 từ khóa tiếng Việt:')
for i, (word, count) in enumerate(vi_keywords, 1):
    bar = '█' * min(count // 50, 30)
    print(f'  {i:2d}. {word:15s} ({count:4d}) {bar}')

# === 4. Nhóm category (tiếng Việt) ===
vi_categories = {
    'Ghế/Bàn': ['ghế', 'bàn', 'kệ', 'tủ', 'giá'],
    'Đèn/Đồ gia dụng': ['đèn', 'bình', 'cốc', 'hộp', 'chai'],
    'Đồ điện tử': ['điện', 'máy', 'laptop', 'màn'],
    'Phương tiện': ['xe', 'ô tô', 'thuyền', 'máy bay'],
    'Gỗ/Vật liệu': ['gỗ', 'kim loại', 'nhựa', 'vải'],
}

print(f'\n--- Phân bố category (tiếng Việt) ---')
for cat, kws in vi_categories.items():
    count = sum(vi_counter.get(kw, 0) for kw in kws)
    print(f'  {cat:20s}: {count:4d} mentions')

print(f'\n✓ Chuẩn hóa caption tiếng Việt hoàn tất!')

## 5. Cấu hình Training (CPU-only)

Config **cố định cho CPU** (12.7 GB RAM, 225.8 GB Disk):
- **MLP decoder** (nhanh hơn transformer ~5x trên CPU)
- **batch_size=8**, num_points=1024, max_objects=5000
- **Tắt EMD** (O(n²) cực chậm trên CPU)
- Save checkpoint mỗi **30 batches** (~36 giây)
- ~10 phút/epoch

In [ ]:
import yaml, os, json
import psutil

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'System RAM: {ram_gb:.1f} GB | CPU-only mode')

# === CONFIG CỐ ĐỊNH CHO CPU (Phase 2: Multi-view + Normal Consistency) ===
colab_config = {
    'data': {
        'data_dir': DATA_DIR,
        'num_points': 1024,
        'image_size': 224,
        'batch_size': 8,
        'num_workers': 4,
        'max_objects': 5000,
        'num_views': 4,          # Phase 2: DiffuRank-inspired 4 views/object
    },
    'model': {
        'clip_model': 'ViT-B-32',
        'pretrained': 'laion2b_s34b_b79k',
        'freeze_clip': True,
        'decoder_type': 'mlp',
        'decoder_width': 256,
        'decoder_layers': 3,
        'decoder_heads': 4,
        'fine_layers': 2,
        'fusion_type': 'concat',
        'fused_dim': 512,
        'coarse_points': 256,
        'fine_points': 1024,
    },
    'training': {
        'epochs': 100,
        'lr': 0.001,
        'weight_decay': 0.01,
        'warmup_epochs': 5,
        'patience': 25,
        'cooldown_seconds': 0,
    },
    'loss': {
        'lambda_cd': 1.0,
        'lambda_emd': 0.0,
        'lambda_color': 0.5,
        'lambda_repulsion': 0.1,
        'lambda_uniform': 0.05,
        'lambda_normal': 0.05,   # Phase 3A: Normal Consistency (Laplacian smoothness)
        'normal_k': 8,
        'repulsion_k': 8,
        'repulsion_h': 0.03,
        'use_emd': False,
    },
    'logging': {
        'log_dir': 'runs',
        'checkpoint_dir': 'checkpoints',
        'log_every': 10,
        'val_every': 1,
        'save_every': 30,
    }
}

config_path = f'{WORK_DIR}/configs/colab.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(colab_config, f, default_flow_style=False)

progress_path = f'{WORK_DIR}/training_progress.json'
if not os.path.exists(progress_path):
    with open(progress_path, 'w') as f:
        json.dump({'status': 'configured', 'config_tier': 'CPU'}, f)

n_obj = 5000
batches = n_obj // 8
print(f'\n✓ Config saved (CPU + Multi-view + Normal Consistency):')
print(f'  decoder=mlp, batch=8, points=1024, epochs=100')
print(f'  num_views=4 (DiffuRank-inspired multi-view)')
print(f'  lambda_normal=0.05 (Laplacian smoothness)')
print(f'  objects=5000, ~{batches} batches/epoch')
print(f'  ⏱ ~{batches * 1.5 / 60:.0f} phút/epoch (4x views tăng compute)')

## 5.5 DiffuRank Best View Precomputation (Phase 2 - Plan.md)

Tính toán **view có CLIP-caption alignment cao nhất** cho mỗi object.  
Chạy **1 lần duy nhất** → lưu `best_views_cache.json` → dataset sẽ tự động dùng khi load.

> **DiffuRank (ECCV 2024)**: thay vì chọn view ngẫu nhiên, chọn view align nhất với caption → features tốt hơn.  
> **Bỏ qua** nếu rendered images chưa có (chỉ dùng ShapeNet point clouds).

In [ ]:
%%time
import os, json, torch
import open_clip
from PIL import Image
from torchvision import transforms

DATA_DIR = f'{WORK_DIR}/Cap3D_data'
cache_path = f'{DATA_DIR}/best_views_cache.json'

if os.path.exists(cache_path):
    with open(cache_path) as f:
        cache = json.load(f)
    print(f'✓ DiffuRank cache đã có ({len(cache)} objects) — bỏ qua')
else:
    img_dir = f'{DATA_DIR}/rendered_images_masked'
    if not os.path.exists(img_dir) or len(os.listdir(img_dir)) == 0:
        img_dir = f'{DATA_DIR}/rendered_images'
    
    if not os.path.exists(img_dir) or len(os.listdir(img_dir)) == 0:
        print('⚠ Không có rendered images → bỏ qua DiffuRank precompute')
        print('  Dataset sẽ dùng random view selection')
    else:
        print(f'Precomputing DiffuRank best views từ {img_dir}...')
        
        # Load CLIP model
        device = 'cpu'
        clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
        clip_model.eval()
        tokenizer = open_clip.get_tokenizer('ViT-B-32')
        
        preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                                 std=[0.26862954, 0.26130258, 0.27577711]),
        ])
        
        # Load captions
        with open(f'{DATA_DIR}/captions.json') as f:
            captions_data = json.load(f)
        
        cache = {}
        obj_dirs = [d for d in os.listdir(img_dir) if os.path.isdir(f'{img_dir}/{d}')]
        
        with torch.no_grad():
            for i, obj_id in enumerate(obj_dirs):
                if obj_id not in captions_data:
                    continue
                
                obj_img_dir = f'{img_dir}/{obj_id}'
                image_files = sorted([
                    f for f in os.listdir(obj_img_dir)
                    if f.endswith('.png') and '_depth' not in f and '_MatAlpha' not in f
                ])
                
                if not image_files:
                    continue
                
                # Lấy caption
                caps = captions_data[obj_id]
                cap_text = caps.get('blip2', caps.get('caption', caps.get('general', '')))
                if isinstance(cap_text, list):
                    cap_text = cap_text[0]
                
                # Encode text
                text_tokens = tokenizer([str(cap_text)])
                text_feat = clip_model.encode_text(text_tokens).float()
                text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
                
                # Encode tất cả views, tính CLIP similarity
                scores = []
                for img_file in image_files:
                    try:
                        img = Image.open(f'{obj_img_dir}/{img_file}').convert('RGB')
                        img_tensor = preprocess(img).unsqueeze(0)
                        img_feat = clip_model.encode_image(img_tensor).float()
                        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
                        score = (img_feat @ text_feat.T).item()
                        scores.append((score, img_file))
                    except Exception:
                        pass
                
                if scores:
                    # Sắp xếp giảm dần theo CLIP score
                    scores.sort(key=lambda x: -x[0])
                    cache[obj_id] = [s[1] for s in scores]  # list files theo thứ tự tốt nhất
                
                if (i + 1) % 500 == 0 or (i + 1) == len(obj_dirs):
                    print(f'  [{i+1}/{len(obj_dirs)}] processed')
                    # Lưu intermediate
                    with open(cache_path, 'w') as f:
                        json.dump(cache, f)
        
        with open(cache_path, 'w') as f:
            json.dump(cache, f)
        
        print(f'\n✓ DiffuRank cache saved: {len(cache)} objects → {cache_path}')
        print('  Dataset sẽ tự động dùng best view khi training')

## 6. Training (CPU)

**Nếu bị ngắt**: Chạy Cell 2 (Mount) → Cell 8 (Deps) → Cell 18 (Config) → Cell 20 (Keep-alive) → Cell 21 (Train)

**Bảo hiểm:**
- Keep-alive JavaScript chống timeout
- Checkpoint lưu mỗi **30 batches** (mất tối đa ~36s)
- Resume tự động phát hiện checkpoint trên Drive
- Atomic save → file không bao giờ bị hỏng

In [ ]:
# === CHỐNG COLAB TIMEOUT ===
from IPython.display import display, Javascript
display(Javascript('''
function KeepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    var buttons = document.querySelectorAll("colab-connect-button");
    if (buttons.length > 0) { buttons[0].click(); }
    var toolbar = document.querySelector("#toolbar");
    if (toolbar) {
        var event = new MouseEvent('click', { bubbles: true });
        toolbar.dispatchEvent(event);
    }
}
setInterval(KeepAlive, 60000);
console.log("✓ Keep-alive started (60s interval)");
'''))
print('✓ Keep-alive đã bật! Session sẽ không bị timeout.')

%%time
# === BẮT ĐẦU TRAINING (AUTO-RESUME) ===
import os, torch, json, time
os.chdir(WORK_DIR)

CKPT_DIR = f'{WORK_DIR}/checkpoints'
resume_path = ''

# Tìm checkpoint tương thích để resume
for ckpt_name in ['last_model.pth', 'best_model.pth']:
    ckpt_file = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(ckpt_file):
        try:
            ckpt = torch.load(ckpt_file, map_location='cpu')
            ckpt_cfg = ckpt.get('config', {})
            ckpt_points = ckpt_cfg.get('model', {}).get('fine_points', 0)
            ckpt_decoder = ckpt_cfg.get('model', {}).get('decoder_type', 'mlp')

            import yaml
            with open(f'{WORK_DIR}/configs/colab.yaml') as f:
                cur_cfg = yaml.safe_load(f)
            cur_points = cur_cfg.get('model', {}).get('fine_points', 0)
            cur_decoder = cur_cfg.get('model', {}).get('decoder_type', 'mlp')

            if ckpt_points == cur_points and ckpt_decoder == cur_decoder:
                resume_path = ckpt_file
                epoch_done = ckpt.get('epoch', -1) + 1
                best_cd = ckpt.get('best_val_cd', float('inf'))
                print(f'✓ Resume từ: {ckpt_name} (epoch {epoch_done}, CD={best_cd:.4f})')
                break
            else:
                print(f'✗ Checkpoint không tương thích ({ckpt_name}) → Xóa')
                os.remove(ckpt_file)
        except Exception as e:
            print(f'✗ Checkpoint hỏng: {ckpt_name} → Xóa ({e})')
            os.remove(ckpt_file)

if not resume_path:
    print('▶ Train từ đầu')

resume_flag = f'--resume "{resume_path}"' if resume_path else ''

# Lưu progress
progress_path = f'{WORK_DIR}/training_progress.json'
with open(progress_path, 'w') as f:
    json.dump({'status': 'training', 'start': time.strftime('%Y-%m-%d %H:%M:%S')}, f)

print(f'\n🏃 Bắt đầu training... (checkpoint lưu mỗi 30 batches)')

!python src/train.py \
    --config configs/colab.yaml \
    {resume_flag}

# Cập nhật progress
with open(progress_path, 'w') as f:
    json.dump({'status': 'completed', 'end': time.strftime('%Y-%m-%d %H:%M:%S')}, f)

# Kiểm tra kết quả
for f_name in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{f_name}'
    if os.path.exists(p):
        size = os.path.getsize(p) / (1024**2)
        print(f'✓ {f_name} ({size:.1f} MB)')
print('\n✓ Training hoàn tất! Checkpoints trên Drive.')

In [ ]:
os.chdir(WORK_DIR)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

!python src/evaluate.py \
    --config configs/colab.yaml \
    --checkpoint checkpoints/best_model.pth \
    --split test \
    --output results/evaluation.json

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
os.chdir(WORK_DIR)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

!python src/evaluate.py \
    --config configs/colab.yaml \
    --checkpoint checkpoints/best_model.pth \
    --split test \
    --output results/evaluation.json

In [ ]:
# Xem kết quả evaluation
import json, os
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
eval_file = f'{WORK_DIR}/results/evaluation.json'
if os.path.exists(eval_file):
    with open(eval_file) as f:
        results = json.load(f)
    print('=== KẾT QUẢ EVALUATION ===')
    for k, v in results.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.6f}')
        else:
            print(f'  {k}: {v}')
else:
    print('⚠ Chưa có evaluation results. Chạy cell trên trước.')

In [ ]:
import os, json, torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === 1. Đọc metrics từ checkpoint ===
CKPT_DIR = f'{WORK_DIR}/checkpoints'
print('=== ĐỘ ĐO ĐÁNH GIÁ ===\n')

# Đọc best checkpoint
best_path = f'{CKPT_DIR}/best_model.pth'
last_path = f'{CKPT_DIR}/last_model.pth'

metrics_summary = {}
for name, path in [('Best model', best_path), ('Last model', last_path)]:
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu')
        epoch = ckpt.get('epoch', -1) + 1
        best_cd = ckpt.get('best_val_cd', float('inf'))
        size_mb = os.path.getsize(path) / (1024**2)
        metrics_summary[name] = {'epoch': epoch, 'best_cd': best_cd, 'size_mb': size_mb}
        print(f'{name}:')
        print(f'  Epoch: {epoch}')
        print(f'  Best Val CD: {best_cd:.6f}')
        print(f'  Size: {size_mb:.1f} MB')
        print()

# === 2. Đọc evaluation results ===
eval_path = f'{WORK_DIR}/results/evaluation.json'
if os.path.exists(eval_path):
    with open(eval_path) as f:
        eval_results = json.load(f)
    print('--- Kết quả Evaluation trên Test Set ---')
    for metric, value in eval_results.items():
        if isinstance(value, float):
            print(f'  {metric:25s}: {value:.6f}')
        else:
            print(f'  {metric:25s}: {value}')
else:
    print('⚠ Chưa có evaluation results (chạy Cell 22 trước)')
    eval_results = {}

# === 3. Đọc TensorBoard logs (nếu có) ===
log_dir = f'{WORK_DIR}/runs'
train_losses = []
val_losses = []

# Thử đọc từ TensorBoard events
try:
    from tensorboard.backend.event_processing import event_accumulator
    
    # Tìm event file
    event_files = []
    if os.path.exists(log_dir):
        for root, dirs, files in os.walk(log_dir):
            for f in files:
                if 'events.out.tfevents' in f:
                    event_files.append(os.path.join(root, f))
    
    if event_files:
        ea = event_accumulator.EventAccumulator(log_dir)
        ea.Reload()
        
        # Đọc train/val CD
        if 'train/chamfer_distance' in ea.Tags()['scalars']:
            for event in ea.Scalars('train/chamfer_distance'):
                train_losses.append({'epoch': event.step, 'cd': event.value})
        if 'val/chamfer_distance' in ea.Tags()['scalars']:
            for event in ea.Scalars('val/chamfer_distance'):
                val_losses.append({'epoch': event.step, 'cd': event.value})
        
        print(f'\n✓ Đọc được {len(train_losses)} epoch train logs, {len(val_losses)} epoch val logs')
except Exception as e:
    print(f'\n⚠ Không đọc được TensorBoard logs: {e}')

# === 4. Trực quan hóa metrics ===
if train_losses or val_losses:
    fig = make_subplots(rows=1, cols=1, subplot_titles=['Chamfer Distance theo Epoch'])
    
    if train_losses:
        fig.add_trace(go.Scatter(
            x=[d['epoch'] for d in train_losses],
            y=[d['cd'] for d in train_losses],
            mode='lines', name='Train CD',
            line=dict(color='dodgerblue', width=2)
        ))
    
    if val_losses:
        fig.add_trace(go.Scatter(
            x=[d['epoch'] for d in val_losses],
            y=[d['cd'] for d in val_losses],
            mode='lines+markers', name='Val CD',
            line=dict(color='orangered', width=2),
            marker=dict(size=4)
        ))
    
    fig.update_layout(
        title='Training Progress - Chamfer Distance',
        xaxis_title='Epoch',
        yaxis_title='Chamfer Distance (lower = better)',
        height=400,
        template='plotly_dark',
    )
    fig.show()
    
    # Hiển thị bảng tóm tắt
    if val_losses:
        best_epoch = min(val_losses, key=lambda x: x['cd'])
        print(f'\n📊 Best epoch: {best_epoch["epoch"]} (Val CD = {best_epoch["cd"]:.6f})')
        print(f'   Last epoch: {val_losses[-1]["epoch"]} (Val CD = {val_losses[-1]["cd"]:.6f})')
else:
    print('\n⚠ Chưa có training logs. Chạy training trước (Cell 21) rồi quay lại đây.')
    print('   Hoặc xem kết quả evaluation ở trên.')

# === 5. Bảng tổng hợp metrics ===
print('\n=== BẢNG TỔNG HỢP ĐỘ ĐO ===')
print(f'{"Metric":<25} {"Giá trị":<15} {"Đánh giá"}')
print('-' * 55)
if eval_results:
    cd = eval_results.get('chamfer_distance', eval_results.get('cd', None))
    fscore = eval_results.get('f_score', eval_results.get('fscore', None))
    if cd is not None:
        quality = '🟢 Tốt' if cd < 0.01 else ('🟡 Trung bình' if cd < 0.05 else '🔴 Cần cải thiện')
        print(f'{"Chamfer Distance":<25} {cd:<15.6f} {quality}')
    if fscore is not None:
        quality = '🟢 Tốt' if fscore > 0.8 else ('🟡 Trung bình' if fscore > 0.5 else '🔴 Cần cải thiện')
        print(f'{"F-Score":<25} {fscore:<15.4f} {quality}')
elif metrics_summary:
    cd = metrics_summary.get('Best model', {}).get('best_cd', None)
    if cd:
        quality = '🟢 Tốt' if cd < 0.01 else ('🟡 Trung bình' if cd < 0.05 else '🔴 Cần cải thiện')
        print(f'{"Best Val CD":<25} {cd:<15.6f} {quality}')

print(f'\n✓ Hiển thị độ đo hoàn tất!')

## 8. Gradio Demo (Interactive UI) - HỖ TRỢ TIẾNG VIỆT

Chạy Gradio demo với:
- Upload ảnh + nhập mô tả **tiếng Việt hoặc tiếng Anh** → tạo 3D
- Ví dụ: "một chiếc ghế gỗ có bốn chân", "một chiếc đèn bàn màu trắng"
- 3 mức chất lượng: low / medium / high
- Xuất .ply, .obj, .glb
- Public link để chia sẻ

**Chỉ cần chạy Cell này để demo!** (sau khi đã mount Drive + install deps)

In [ ]:
# === GRADIO DEMO (hỗ trợ tiếng Việt) ===
# Chỉ cần chạy cell này sau khi đã:
#   1. Mount Drive (Cell 2)
#   2. Install deps (Cell 8)
#   3. Có checkpoint trên Drive (checkpoints/best_model.pth)

import os, sys
WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

# Chạy Gradio demo với public link
# UI hỗ trợ tiếng Việt: ví dụ "một chiếc ghế gỗ có bốn chân"
!python src/demo.py \
    --config configs/colab.yaml \
    --checkpoint checkpoints/best_model.pth \
    --share

## 9. Download Checkpoint về máy

Tải checkpoint đã train về máy để chạy demo local.

In [ ]:
# Download checkpoint về máy
from google.colab import files
import os

WORK_DIR = '/content/drive/MyDrive/ThiGiacMayTinh_BaiTapLon'
CKPT_DIR = f'{WORK_DIR}/checkpoints'

for ckpt_name in ['best_model.pth', 'last_model.pth']:
    p = f'{CKPT_DIR}/{ckpt_name}'
    if os.path.exists(p):
        size = os.path.getsize(p) / (1024**2)
        print(f'✓ {ckpt_name} ({size:.1f} MB)')

# Download best checkpoint
ckpt_to_download = f'{CKPT_DIR}/best_model.pth'
if not os.path.exists(ckpt_to_download):
    ckpt_to_download = f'{CKPT_DIR}/last_model.pth'

if os.path.exists(ckpt_to_download):
    print(f'\nDownloading {os.path.basename(ckpt_to_download)}...')
    files.download(ckpt_to_download)
else:
    print('✗ Không tìm thấy checkpoint nào!')

## 10. Ghi chú & Hướng dẫn

### Cấu trúc dự án trên Drive:
```
ThiGiacMayTinh_BaiTapLon/
├── src/
│   ├── train.py, evaluate.py, demo.py
│   ├── data/ (dataset.py, prepare_data.py, transforms.py)
│   ├── models/ (multimodal3d.py, encoders.py, decoder.py, fusion.py)
│   └── losses/ (losses.py)
├── configs/
│   └── colab.yaml           # Config CPU-only (auto-generated)
├── Cap3D_data/
│   ├── pointclouds/         # ABO point clouds (.ply)
│   ├── rendered_images/     # ABO 20 views + depth + mask
│   ├── rendered_images_masked/  # Ảnh đã lọc background
│   ├── shapenet/pointclouds/    # ShapeNet point clouds (.ply)
│   ├── Cap3D_automated_ABO.csv
│   ├── Cap3D_automated_ShapeNet.csv
│   ├── captions.json
│   ├── captions_vi.json     # Captions tiếng Việt
│   └── splits/ (train.txt, val.txt, test.txt)
├── checkpoints/             # auto-save lên Drive
├── results/
└── training_progress.json
```

### QUICK RESUME sau khi bị ngắt:
```
1. Mở lại Colab
2. Chạy: Cell 2 (Mount) → Cell 8 (Deps) → Cell 18 (Config) → Cell 20 (Keep-alive) → Cell 21 (Train)
3. Training tự resume từ checkpoint trên Drive
```

### CHỈ DEMO (không train):
```
1. Cell 2 (Mount) → Cell 8 (Deps) → Cell 27 (Gradio Demo)
```

### Cấu hình CPU (cố định):
| Tham số | Giá trị | Lý do |
|---|---|---|
| decoder_type | **mlp** | Nhanh hơn transformer ~5x trên CPU |
| batch_size | **8** | RAM 12.7 GB đủ |
| num_points | **1024** | Giảm 50% compute, vẫn đủ shape |
| max_objects | **5000** | ~10 phút/epoch |
| save_every | **30 batches** | ~36s progress tối đa |

### Những cải thiện đã áp dụng (từ Plan.md):
- ✅ **MatAlpha mask** lọc background → CLIP focus vào object
- ✅ **ShapeNet data** thêm 52k objects → model generalise tốt hơn
- ✅ **Captions tiếng Việt** → demo hỗ trợ song ngữ
- ✅ **Hiển thị độ đo** (CD, F-Score) → đánh giá trực quan
- ✅ **CPU-optimized** config → epoch nhanh, save thường xuyên